# 🤖 SocialNav3D – SIEP 社交导航演示

> **在启智 (OpenI) 平台上一键运行 SIEP 导航仿真**

本 Notebook 会自动完成以下步骤：
1. 从 GitHub 拉取最新代码
2. 安装所需依赖
3. 运行 SIEP 仿真（无需图形界面）
4. 在 Notebook 内显示轨迹图

**使用方法**：顶部菜单 → `Run` → `Run All Cells`，然后等待几分钟即可。

## 第 1 步：获取代码

如果当前目录里已经有代码（例如你是在项目目录里打开的 Notebook），跳过这一格也没问题。  
否则，运行下面这格会从 GitHub 克隆仓库。

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/PENGYAN-lang/Socially-Acceptable-Robot-Navigation-in-UnstructuredEnvironments-SIEP-concept-.git"
REPO_DIR = "SocialNav3D"   # 克隆到当前目录下的 SocialNav3D 子文件夹

def _run(cmd, cwd=None):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if r.stdout: print(r.stdout)
    if r.stderr: print(r.stderr, file=sys.stderr)
    return r

# ── 检测是否已在项目根目录（social_nav3d 包已存在）──────────────────────
if os.path.isdir("social_nav3d") and os.path.isfile("run_demo.py"):
    print("✅ 已在项目根目录，无需克隆。")
elif os.path.isdir(REPO_DIR):
    print(f"目录 {REPO_DIR} 已存在，执行 git pull 更新…")
    _run("git pull", cwd=REPO_DIR)
    os.chdir(REPO_DIR)
else:
    print("开始克隆仓库，请稍候（约 30-60 秒）…")
    _run(f"git clone {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)

# ── 确保 social_nav3d 包在 Python 搜索路径中 ──────────────────────────
cwd = os.getcwd()
if cwd not in sys.path:
    sys.path.insert(0, cwd)

print("当前目录：", cwd)
print("目录内容：", [x for x in os.listdir(".") if not x.startswith(".")])


## 第 2 步：安装依赖

运行一次即可；之后如果内核没有重启就不需要重复安装。

In [ ]:
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# 核心依赖
pip("pybullet==3.2.6", "numpy>=1.24", "matplotlib>=3.7",
    "scipy>=1.10", "PyYAML>=6.0", "imageio>=2.31", "imageio-ffmpeg>=0.4.9")

print("✅ 依赖安装完成")

## 第 3 步：验证导入

确认所有模块都可以正常导入。

In [ ]:
import importlib, sys

needed = ["pybullet", "numpy", "matplotlib", "scipy", "yaml"]
ok = True
for m in needed:
    try:
        importlib.import_module(m)
        print(f"  ✅ {m}")
    except ImportError as e:
        print(f"  ❌ {m}: {e}")
        ok = False

from social_nav3d.env.sim import SocialNavSim
from social_nav3d.planners.siep_planner import SIEPPlanner
from social_nav3d.utils.config import load_config
print("  ✅ social_nav3d (SIEP planner)")

if ok:
    print("\n所有依赖正常，可以继续运行仿真。")

## 第 4 步：运行 SIEP 仿真

下面的代码会在 **无界面（headless）** 模式下运行仿真，适合服务器环境。  
仿真完成后，轨迹图会保存到 `runs/trajectory_siep.png`。

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")   # 无 GUI 模式，适合服务器
import matplotlib.pyplot as plt
from pathlib import Path

from social_nav3d.env.sim import SocialNavSim
from social_nav3d.planners.siep_planner import SIEPPlanner
from social_nav3d.utils.config import load_config

# ── 读取 SIEP 配置 ──────────────────────────────────────────────────────
cfg = load_config("social_nav3d/configs/siep_demo.yaml")
cfg["sim"]["gui"] = False           # 强制 headless
cfg["sim"]["out_dir"] = "runs"      # 输出目录

# ── 初始化仿真 & 规划器 ──────────────────────────────────────────────────
print("正在初始化仿真环境…")
sim = SocialNavSim(cfg)
sim.reset()
planner = SIEPPlanner(cfg)
print("初始化完成。")

# ── 主循环 ──────────────────────────────────────────────────────────────
print("开始仿真…")
traj = []
for k in range(sim.max_steps):
    state = sim.get_state()
    ped_states = sim.get_pedestrians_state()
    peds = [(d["xy"], d["yaw"], d["vel"], d["ps"]) for d in ped_states]
    lidar_dists, lidar_angles = sim.get_lidar_directional_2d()
    v, w = planner.plan(state, sim.goal_xy, sim.dt, lidar_dists, lidar_angles, peds)
    sim.step(v, w)
    traj.append([state.x, state.y])
    if sim.reached_goal():
        print(f"✅ 机器人到达目标！步数 = {k}，共 {k * sim.dt:.1f} 秒")
        break
    if k % 200 == 0 and k > 0:
        print(f"   步数 {k}/{sim.max_steps}，当前位置 ({state.x:.2f}, {state.y:.2f})")

sim.close()

# ── 保存轨迹图 ───────────────────────────────────────────────────────────
out_dir = Path("runs")
out_dir.mkdir(exist_ok=True)

traj = np.asarray(traj)
fig, ax = plt.subplots(figsize=(7, 7))
sim.plot_topdown(ax)
ax.plot(traj[:, 0], traj[:, 1], linewidth=2, label="机器人轨迹")
ax.scatter([traj[0, 0]], [traj[0, 1]], s=80, marker="o", zorder=5, label="起点")
ax.scatter([sim.goal_xy[0]], [sim.goal_xy[1]], s=120, marker="*",
           color="gold", zorder=5, label="终点")
ax.legend()
ax.set_title("SocialNav3D – SIEP 仿真轨迹")
fig.tight_layout()
out_path = out_dir / "trajectory_siep.png"
fig.savefig(out_path, dpi=160)
plt.close(fig)
print(f"轨迹图已保存：{out_path}")

## 第 5 步：查看结果

下面直接在 Notebook 内显示轨迹图。

In [ ]:
from IPython.display import Image, display
display(Image("runs/trajectory_siep.png", width=600))

## 第 6 步：调参（可选）

修改下面的字典然后重新运行仿真格，观察行为变化。

| 参数 | 含义 | 建议范围 |
|---|---|---|
| `k_goal` | 目标吸引力（巡航速度）| 0.5 – 2.0 |
| `k_obs` | 障碍物排斥力 | 1.0 – 5.0 |
| `k_ps` | 行人个人空间排斥力 | 0.5 – 4.0 |
| `ps_horizon` | 行人预测时域（秒）| 0.5 – 3.0 |
| `k_align` | 速度对齐力（跟随人群）| 0.0 – 1.0 |


In [ ]:
# ── 修改这里来调参 ──────────────────────────────────────────────────────
custom_params = {
    "k_goal":     1.0,    # 目标吸引力
    "k_obs":      2.5,    # 障碍物排斥
    "k_ps":       2.0,    # 个人空间排斥
    "ps_horizon": 1.5,    # 预测时域 [s]
    "k_align":    0.3,    # 速度对齐
}

# ── 重新运行仿真 ─────────────────────────────────────────────────────────
cfg2 = load_config("social_nav3d/configs/siep_demo.yaml")
cfg2["sim"]["gui"] = False
cfg2["planner"].update(custom_params)

sim2 = SocialNavSim(cfg2)
sim2.reset()
planner2 = SIEPPlanner(cfg2)

traj2 = []
for k in range(sim2.max_steps):
    state = sim2.get_state()
    ped_states = sim2.get_pedestrians_state()
    peds = [(d["xy"], d["yaw"], d["vel"], d["ps"]) for d in ped_states]
    lidar_dists, lidar_angles = sim2.get_lidar_directional_2d()
    v, w = planner2.plan(state, sim2.goal_xy, sim2.dt, lidar_dists, lidar_angles, peds)
    sim2.step(v, w)
    traj2.append([state.x, state.y])
    if sim2.reached_goal():
        print(f"✅ 到达目标，步数 = {k}")
        break

sim2.close()

traj2 = np.asarray(traj2)
fig, ax = plt.subplots(figsize=(7, 7))
sim2.plot_topdown(ax)
ax.plot(traj2[:, 0], traj2[:, 1], linewidth=2, color="tab:orange", label="自定义参数轨迹")
ax.scatter([traj2[0, 0]], [traj2[0, 1]], s=80, marker="o", zorder=5, label="起点")
ax.scatter([sim2.goal_xy[0]], [sim2.goal_xy[1]], s=120, marker="*",
           color="gold", zorder=5, label="终点")
ax.legend()
ax.set_title("自定义参数 SIEP 轨迹")
fig.tight_layout()
out_path2 = Path("runs") / "trajectory_custom.png"
fig.savefig(out_path2, dpi=160)
plt.close(fig)
display(Image(str(out_path2), width=600))

## 第 7 步：与原始采样 MPC 对比（可选）

运行对比实验，对比两种规划器的轨迹路径。

In [ ]:
from social_nav3d.planners.sampling_mpc import SamplingMPC

cfg_mpc = load_config("social_nav3d/configs/default.yaml")
cfg_mpc["sim"]["gui"] = False

sim_mpc = SocialNavSim(cfg_mpc)
sim_mpc.reset()
planner_mpc = SamplingMPC(cfg_mpc)

traj_mpc = []
for k in range(sim_mpc.max_steps):
    state = sim_mpc.get_state()
    ped_states = sim_mpc.get_pedestrians_state()
    peds = [(d["xy"], d["yaw"], d["vel"], d["ps"]) for d in ped_states]
    lidar_min = sim_mpc.lidar_min_distance()
    v, w = planner_mpc.plan(state, sim_mpc.goal_xy, sim_mpc.dt, lidar_min, peds)
    sim_mpc.step(v, w)
    traj_mpc.append([state.x, state.y])
    if sim_mpc.reached_goal():
        print(f"MPC 到达目标，步数 = {k}")
        break

sim_mpc.close()

# 绘制对比图
traj_mpc = np.asarray(traj_mpc)
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, t, title, color in [
    (axes[0], traj,     "SIEP 规划器",     "tab:blue"),
    (axes[1], traj_mpc, "采样 MPC 规划器", "tab:red"),
]:
    sim.plot_topdown(ax)   # same world layout
    ax.plot(t[:, 0], t[:, 1], linewidth=2, color=color)
    ax.scatter([t[0, 0]], [t[0, 1]], s=80, marker="o", zorder=5)
    ax.scatter([sim.goal_xy[0]], [sim.goal_xy[1]], s=120, marker="*",
               color="gold", zorder=5)
    ax.set_title(f"{title}\n（路径长度 {len(t)} 步）")

fig.suptitle("SIEP vs 采样 MPC – 轨迹对比", fontsize=14)
fig.tight_layout()
cmp_path = Path("runs") / "comparison.png"
fig.savefig(cmp_path, dpi=160)
plt.close(fig)
display(Image(str(cmp_path), width=900))